# LACRIMAE — F02 VISIO
> *"Le Visionnaire contemple. La forme prend sens avant le rendu."*

**Mission** : Viewer HTML interactif — preview du timing, ajustement de la config, production de `creative_config.json`

**Prérequis** :
- `F02_VISIO/IN/timing.json` — copié depuis F01 CANTOR
- `F02_VISIO/IN/images/` — images copiées depuis SHARED
- `LAC_CUSTOS.py` présent dans `DRIVE_LACRIMAE/`

---

## ÉTAPE 1 — Montage Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive monté.')

## ÉTAPE 2 — Vérification des inputs

In [ ]:
from pathlib import Path
import json

DRIVE_BASE   = Path('/content/drive/MyDrive/DRIVE_LACRIMAE')
F02_BASE     = DRIVE_BASE / 'F02_VISIO'
IN_DIR       = F02_BASE / 'IN'
OUT_DIR      = F02_BASE / 'OUT'
CODEBASE_DIR = F02_BASE / 'CODEBASE'
CUSTOS_PATH  = DRIVE_BASE / 'LAC_CUSTOS.py'

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Validation LAC_CUSTOS check-in
import subprocess, shutil
shutil.copy(CUSTOS_PATH, '/content/LAC_CUSTOS.py')
result = subprocess.run(
    ['python', '/content/LAC_CUSTOS.py', '--frigate', 'F02', '--mode', 'check-in', '--drive-base', str(DRIVE_BASE)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('[VISIO] ERREUR check-in — corriger avant de continuer.')
    print(result.stderr)
else:
    timing_path = IN_DIR / 'timing.json'
    with open(timing_path, 'r') as f:
        timing = json.load(f)
    print(f'Timing chargé : {len(timing["words"])} mots, {timing["audio_duration_s"]}s')

## ÉTAPE 3 — Installation Flask

In [ ]:
!pip install flask -q
print('Flask installé.')

## ÉTAPE 4 — Démarrage du serveur VISIO

In [ ]:
import shutil, threading, os
from pathlib import Path

# Copier les scripts dans /content
shutil.copy(CODEBASE_DIR / 'lac_f02_flask.py', '/content/lac_f02_flask.py')
shutil.copy(CODEBASE_DIR / 'lac_f02_viewer.html', '/content/lac_f02_viewer.html')

# Variables d'environnement
os.environ['DRIVE_BASE'] = str(DRIVE_BASE)

import sys
sys.path.insert(0, '/content')
from lac_f02_flask import app, start_server

PORT = 5000
server_thread = threading.Thread(target=lambda: start_server(PORT), daemon=True)
server_thread.start()

import time; time.sleep(2)
print(f'[VISIO] Serveur Flask démarré.')

## ÉTAPE 5 — Ouverture du viewer via port Colab

In [ ]:
from google.colab.output import eval_js
proxy_url = eval_js(f'google.colab.kernel.proxyPort({PORT})')
print(f'[VISIO] Viewer disponible à : {proxy_url}')
print('[VISIO] Ouvrir ce lien dans un nouvel onglet.')
from IPython.display import HTML
display(HTML(f'<a href="{proxy_url}" target="_blank" style="font-size:16px;color:#c9a84c">→ Ouvrir VISIO Viewer</a>'))

## ÉTAPE 6 — Validation et sauvegarde

```
Dans le viewer :
1. Parcourir la timeline mot par mot
2. Vérifier les images en preview
3. Ajuster les sliders (grain, contrast, brightness, sepia)
4. Ajuster le cut interval (frames par image)
5. Cliquer SCELLER LA CONFIG
```

## ÉTAPE 7 — Validation LAC_CUSTOS check-out

In [ ]:
!python /content/LAC_CUSTOS.py --frigate F02 --mode check-out --drive-base "{DRIVE_BASE}"

## ÉTAPE 8 — Instructions de transit

```
✓ Si LAC_CUSTOS a validé :

  Copier manuellement :
  F02_VISIO/OUT/creative_config.json  →  F03_PICTOR/IN/creative_config.json

  Puis inscrire le transit dans TRACKING/LACRIMAE_TRANSFER_LOG.md
```